In [ ]:
import os
import cv2
import random
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)
print("PyTorch Version:", torch.__version__)


In [ ]:
valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

authentic_dirs = [

    # CASIA
    '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2/Au',

    # Columbia
    '/kaggle/input/datasets/shriya0/columbia/Columbia Uncompressed Image Splicing Detection/4cam_auth/4cam_auth',


    # IMD2020
    '/kaggle/input/datasets/saikotahmed/imd2020/IMD2020/Authentic'

]


tampering_dirs = [

    # CASIA
    '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2/Tp',

    # Columbia
    '/kaggle/input/datasets/shriya0/columbia/Columbia Uncompressed Image Splicing Detection/4cam_splc/4cam_splc',


    # IMD2020
    '/kaggle/input/datasets/saikotahmed/imd2020/IMD2020/img'


]

def collect_image_paths(dirs):
    paths = []
    for d in dirs:
        if not os.path.exists(d):
            print(f"[WARNING] Không tìm thấy thư mục: {d}")
            continue
        files = [
            os.path.join(d, f)
            for f in os.listdir(d)
            if f.lower().endswith(valid_exts)
        ]
        print(f"{d} → {len(files)} ảnh")
        paths.extend(files)
    return paths

all_authentic_paths = collect_image_paths(authentic_dirs)
all_tampering_paths = collect_image_paths(tampering_dirs)

authentic_paths = all_authentic_paths
tampering_paths = all_tampering_paths
print(f"[Raw] Ảnh Gốc (Nhãn 0) : {len(all_authentic_paths)}")
print(f"[Raw] Ảnh Giả (Nhãn 1) : {len(all_tampering_paths)}")


In [ ]:
class TamperingDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Đọc ảnh
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
        
        
        img = Image.fromarray(img)
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.long)

# Chia dữ liệu
all_paths = [(path, 0) for path in authentic_paths] + \
            [(path, 1) for path in tampering_paths]

paths = [p for p, _ in all_paths]
labels = [l for _, l in all_paths]

train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=42
)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=0.125,
    stratify=train_labels, random_state=42
)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_dataset = TamperingDataset(train_paths, train_labels, transform=train_transform)
val_dataset = TamperingDataset(val_paths, val_labels, transform=val_transform)
test_dataset = TamperingDataset(test_paths, test_labels, transform=val_transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=None)
# Chỉ dùng kênh RGB 
model.conv1 = nn.Conv2d(
    3, 64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) 

if torch.cuda.device_count() > 1:
    print(f"Đang sử dụng {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model) # Bọc model lại bằng DataParallel

model = model.to(device)

print(model)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

num_epochs = 200
best_val_loss = float('inf')
patience = 5
patience_counter = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
        
    train_loss = running_loss / len(train_dataset)
    train_acc = correct / total
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    val_loss = val_loss / len(val_dataset)
    val_acc = correct / total
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
        scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_weights.pth')
        print(f"Validation loss improved. Saved model to best_model_weights.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break


In [ ]:
model.load_state_dict(torch.load('best_model_weights.pth'))
model.eval()

y_true = []
y_pred_probs = []
y_pred_classes = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        _, predicted = outputs.max(1)
        
        y_true.extend(labels.cpu().numpy())
        y_pred_probs.extend(probs.cpu().numpy())
        y_pred_classes.extend(predicted.cpu().numpy())

print("Classification Report:")
print(classification_report(y_true, y_pred_classes))

auc_score = roc_auc_score(y_true, y_pred_probs)
print(f"AUC Score: {auc_score:.4f}")

cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Authentic (0)', 'Tampered (1)'],
            yticklabels=['Authentic (0)', 'Tampered (1)'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Plot training & validation curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.title('Loss over Epochs')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.legend()
plt.title('Accuracy over Epochs')
plt.show()
